# Train Mamba on Z24 — fixed setup holdout

This notebook uses the same setup split for every class: setups 0–4 for training, 5–6 for validation, and 7–8 for testing. This avoids the class–setup confounding in the legacy rotating `balanced` split. Metrics and setup analysis use independent recordings as the primary evaluation unit. Enable **GPU** and **Internet** in Kaggle before running.

In [ ]:
%cd /kaggle/working
!rm -rf time-series-project
!git clone https://github.com/TuanHungGiang/time-series-project.git
%cd time-series-project
!pip install -q huggingface_hub scikit-learn matplotlib

from huggingface_hub import snapshot_download

DATA_DIR = snapshot_download(
    repo_id="thanglexuan/Z24-dataset-processed",
    repo_type="dataset",
    local_dir="Z24-dataset-processed",
    allow_patterns=["inputs.npy", "labels.npy"],
)
print("Dataset saved at:", DATA_DIR)

The strict run enables train-only augmentation and stronger regularization. The run name matches the actual batch size.

In [ ]:
!python -u train_mamba.py \
  --split-mode setup_holdout \
  --epochs 60 \
  --patience 12 \
  --batch-size 16 \
  --learning-rate 3e-4 \
  --weight-decay 1e-3 \
  --dropout 0.3 \
  --num-workers 4 \
  --run-name mamba_setup_holdout_aug_bs16_seed42

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

run_dir = Path("results/mamba_setup_holdout_aug_bs16_seed42")
metrics = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2))

for name in [
    "training_curves.png",
    "confusion_matrix.png",
    "accuracy_per_class.png",
    "accuracy_per_class_by_setup.png",
    "roc_curve.png",
    "tsne_epoch_001/tsne_2d.png",
    "tsne_epoch_final/tsne_2d.png",
]:
    display(Image(filename=str(run_dir / name)))

## Recording-level results by class and setup

Each row below is one independent source recording. Do not use `predictions_test.npz` here: that file contains six correlated windows per recording.

In [ ]:
import numpy as np
import pandas as pd

pred = np.load(run_dir / "predictions_test_recording.npz")
df_pred = pd.DataFrame({
    "true_class": pred["y_true"],
    "predicted_class": pred["y_pred"],
    "setup": pred["setup"],
})
df_pred["correct"] = (df_pred["true_class"] == df_pred["predicted_class"]).astype(int)

accuracy_by_setup = (
    df_pred.groupby(["true_class", "setup"])["correct"]
    .agg(correct="sum", recordings="count", accuracy="mean")
    .reset_index()
)
accuracy_by_setup["accuracy_percent"] = 100 * accuracy_by_setup["accuracy"]
display(accuracy_by_setup)

pivot_accuracy = accuracy_by_setup.pivot(
    index="true_class", columns="setup", values="accuracy_percent"
)
display(pivot_accuracy.round(2))